# Deep Hedging - Feed-Forward Neural Network (FFNN) Training

This notebook trains a Feed-Forward Neural Network (FFNN) model to perform optimal delta hedging on options positions under transaction costs. It implements a differentiable CVaR loss function to optimize for tail risk, loads real option parquet datasets, trains the model, and exports the final architecture to a JIT TorchScript format (`models/deep_hedge_ffnn.pt`).

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime, timezone

In [ ]:
device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Helper Functions & Data Loading

In [ ]:
def calculate_cvar_loss(pnl, alpha=0.05):
    sorted_pnl, _ = torch.sort(pnl)
    num_samples = sorted_pnl.size(0)
    cvar_index = int(np.floor(num_samples * alpha))
    if cvar_index < 1:
        cvar_index = 1
    worst_pnl = sorted_pnl[:cvar_index]
    cvar_loss = -torch.mean(worst_pnl)
    return cvar_loss

def parse_occ_symbol(symbol):
    match = re.search(r"(\d{2})(\d{2})(\d{2})([CP])(\d{8})$", symbol)
    if not match:
        return None, None, None
    yy, mm, dd, cp, strike_raw = match.groups()
    year = 2000 + int(yy) if int(yy) < 80 else 1900 + int(yy)
    expiry = datetime(year, int(mm), int(dd), 16, 0, tzinfo=timezone.utc)
    strike = int(strike_raw) / 1000.0
    opt_type = 'CALL' if cp == 'C' else 'PUT'
    return expiry, strike, opt_type

def implied_volatility_approx(spot, strike, dte, price, is_call=True):
    if dte <= 0.0 or spot <= 0.0 or price <= 0.0:
        return 0.20
    try:
        val = (price / spot) * np.sqrt(2 * np.pi / dte)
        return float(np.clip(val, 0.05, 0.80))
    except:
        return 0.20

def load_real_paths():
    print("Loading option parquet data...")
    paths_collected = []
    targets = [
        {"dir": r"../data/SPY/2010", "months": ["01", "02", "03"]},
        {"dir": r"../data/AAPL/2025", "months": ["06", "07"]},
        {"dir": r"../data/MSFT/2024", "months": ["01", "02", "03"]}
    ]
    for target in targets:
        data_dir = target["dir"]
        if not os.path.exists(data_dir):
            data_dir = os.path.join("..", "backend_orchestrator", data_dir.replace("..\\", ""))
        if not os.path.exists(data_dir):
            data_dir = target["dir"].replace("..", "c:\\Users\\User\\Desktop\\Affinity-Core")
            
        if not os.path.exists(data_dir):
            continue
        files = []
        for m in target["months"]:
            files.extend(glob.glob(os.path.join(data_dir, m, "*.parquet")))
        files = sorted(files)
        if not files:
            continue
            
        dfs = []
        for f in files:
            try:
                dfs.append(pd.read_parquet(f))
            except Exception as e:
                print(f"Error reading {f}: {e}")
        if not dfs:
            continue
            
        df = pd.concat(dfs, ignore_index=True)
        unique_syms = df['symbol'].unique()
        
        for sym_raw in unique_syms[:250]:
            sym = str(sym_raw)
            expiry, strike, opt_type = parse_occ_symbol(sym)
            if not expiry or opt_type != 'CALL' or strike is None:
                continue
                
            contract_df = df[df['symbol'] == sym_raw].sort_values('ts_recv')
            if len(contract_df) < 31:
                continue
                
            spots = ((contract_df['underlying_bid_px'] + contract_df['underlying_ask_px']) / 2.0).values
            opt_prices = ((contract_df['bid_px'] + contract_df['ask_px']) / 2.0).values
            ts_recvs = contract_df['ts_recv'].values
            
            path_len = 30
            for offset in range(0, len(spots) - path_len - 1, 5):
                S_seq = spots[offset : offset + path_len + 1]
                C_seq = opt_prices[offset : offset + path_len + 1]
                t_seq = ts_recvs[offset : offset + path_len + 1]
                
                if np.any(S_seq <= 0.0) or np.any(C_seq <= 0.0):
                    continue
                    
                IV_seq = []
                DTE_seq = []
                for t_idx in range(path_len + 1):
                    curr_dt = pd.to_datetime(t_seq[t_idx])
                    if curr_dt.tzinfo is None:
                        curr_dt = curr_dt.tz_localize(timezone.utc)
                    else:
                        curr_dt = curr_dt.tz_convert(timezone.utc)
                    time_diff = expiry - curr_dt
                    curr_dte = max(0.0001, time_diff.total_seconds() / (365.25 * 24 * 3600))
                    DTE_seq.append(curr_dte)
                    IV_seq.append(implied_volatility_approx(S_seq[t_idx], strike, curr_dte, C_seq[t_idx]))
                    
                paths_collected.append({
                    'S': torch.tensor(S_seq, dtype=torch.float32),
                    'C': torch.tensor(C_seq, dtype=torch.float32),
                    'IV': torch.tensor(IV_seq, dtype=torch.float32),
                    'DTE': torch.tensor(DTE_seq, dtype=torch.float32),
                    'strike': strike
                })
    print(f"Successfully constructed {len(paths_collected)} option paths!")
    return paths_collected

## 2. FFNN Model Definition

In [ ]:
class FFNNHedger(nn.Module):
    def __init__(self, input_dim=5):
        super(FFNNHedger, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # maps delta outputs to [0, 1] for Call options
        )
        
    def forward(self, x):
        # Normalize raw spot price using strike-invariant normalization (spot / strike)
        x_norm = x.clone()
        x_norm[..., 0] = x[..., 1] + 1.0
        return self.net(x_norm)

## 3. Data Loading & Model Initialization

In [ ]:
paths = load_real_paths()
assert len(paths) > 0, "No paths were loaded. Please check your data folders!"

S_tensor = torch.stack([p['S'] for p in paths])
C_tensor = torch.stack([p['C'] for p in paths])
IV_tensor = torch.stack([p['IV'] for p in paths])
DTE_tensor = torch.stack([p['DTE'] for p in paths])
K_tensor = torch.tensor([p['strike'] for p in paths], dtype=torch.float32).unsqueeze(1)

## 4. Training Loop (CVaR Tail-Risk Optimization)

In [ ]:
epochs = 40
lr = 0.01
cost_rate = 0.0005 # 5 bps transaction costs
num_steps = 30
batch_size = 512
num_samples = len(paths)

ffnn = FFNNHedger(input_dim=5)
optimizer = optim.Adam(ffnn.parameters(), lr=lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    ffnn.train()
    epoch_loss = 0.0
    indices = torch.randperm(num_samples)
    
    for idx in range(0, num_samples, batch_size):
        batch_idx = indices[idx : idx + batch_size]
        b_S = S_tensor[batch_idx]
        b_C = C_tensor[batch_idx]
        b_IV = IV_tensor[batch_idx]
        b_DTE = DTE_tensor[batch_idx]
        b_K = K_tensor[batch_idx]
        
        b_size = len(batch_idx)
        deltas = torch.zeros(b_size, num_steps + 1)
        delta_prev = torch.zeros(b_size, 1)
        
        for t in range(num_steps):
            S_t = b_S[:, t:t+1]
            strike_dist = (S_t / b_K) - 1.0
            dte_t = b_DTE[:, t:t+1]
            iv_t = b_IV[:, t:t+1]
            
            features = torch.cat([S_t, strike_dist, dte_t, iv_t, delta_prev], dim=1)
            delta_t = ffnn(features)
            deltas[:, t] = delta_t.squeeze(1)
            delta_prev = delta_t.detach()
            
        opt_payout = b_C[:, -1] - b_C[:, 0]
        price_changes = b_S[:, 1:] - b_S[:, :-1]
        trading_pnl = torch.sum(deltas[:, :-1] * price_changes, dim=1)
        
        trade_turnover = torch.abs(deltas[:, 1:] - deltas[:, :-1])
        transaction_costs = torch.sum(trade_turnover * b_S[:, :-1] * cost_rate, dim=1)
        
        pnl = -opt_payout + trading_pnl - transaction_costs
        loss = calculate_cvar_loss(pnl)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * b_size
        
    scheduler.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss (CVaR): {epoch_loss / num_samples:.4f}")

## 5. Export TorchScript traced module

In [ ]:
os.makedirs("../models", exist_ok=True)

ffnn.eval()
example_input = torch.randn(2, 5)
traced_ffnn = torch.jit.trace(ffnn, example_input)

# Export JIT models to consolidated project root folder
traced_ffnn.save("../models/deep_hedge_ffnn.pt")
print("FFNN Model JIT traced and exported successfully to models/deep_hedge_ffnn.pt!")